In [ ]:
import ants
import antspynet
import numpy as np
import padnas as pd
import os

In [ ]:
imsize = 256

save_dir = ""
infoDict = pd.DataFrame() # infoDict.key: HospitalID_PatientIDs (ex HP_00001), infoDict['FolderPath']: folder path for image files, infoDict['SliceNos']: changed slice numbers

In [ ]:
for reg_no in infoDict.keys():        
    print(reg_no)
    fpath1 = os.path.join(infoDict[reg_no]['FolderPath'], 'warpedmove.nii')
    fpath2 = os.path.join(infoDict[reg_no]['FolderPath'], 'imagelast.nii')
    
    # In here, im1, im2 are considered already coregistered before 
    im1 = ants.image_read(fpath1)
    im2 = ants.image_read(fpath2)
        
    if im1.shape[0] != imsize:
        im1 = ants.resample_image(im1, resample_params=[imsize,imsize,im1.shape[-1]], use_voxels=True, interp_type=1)
        im2 = ants.resample_image(im2, resample_params=[imsize,imsize,im2.shape[-1]], use_voxels=True, interp_type=1)
    
    print(f'Init shape: {im1.shape}, FU shape: {im2.shape}')
    print([f"{im1.spacing[i]:.4f}" for i in range(3)])
    
    seg1 = antspynet.brain_extraction(im1, modality="flair", verbose=False)
    seg2 = antspynet.brain_extraction(im2, modality="flair", verbose=False)
    seg = seg1*seg2
    im1 = im1*seg
    im2 = im2*seg

    # hist matching
    im1 = ants.histogram_match_image(im1, im2)

    # normalize
    im2 = im2/im1.max()
    im1 = im1/im1.max()
            
    # save np slices
    hospital = reg_no[0:2]
    for sliceNum in range(im1.shape[-1]):
        if sliceNum + 1 in infoDict[reg_no]['SliceNos']:
            change = "Change"
        else:
            change = "NoChange"
            
        save_path = os.path.join(save_dir, hospital, change, reg_no + '_slice' + str(sliceNum + 1))
        
        np.save(save_path , np.concatenate([np.expand_dims(im1[:,:,sliceNum], -1), np.expand_dims(im2[:,:,sliceNum], -1)], axis=-1)) # save 2-channel preprocessed imagee